# Ollama and the autoregressive loop



## Setup




In [1]:
!date


Tue Jul 28 12:49:40 AM UTC 2026


### Install packages


In [2]:
%%capture output_system_install
%%bash

# bash setup
fgrep -q '.bash_aliases' ~/.bashrc || {
  echo -e '\n\n\n[ -f ~/.bash_aliases ] && source ~/.bash_aliases\n' >> ~/.bashrc
}

{ cat << 'eof'
alias cls='clear'
alias dir='ls -la'
alias goTo='cd '
alias goUp='cd ..'
alias whereAmI='pwd'

alias me.group.id='id -g'
alias me.group.name='id -g -n'
alias me.groups='id -G'
alias me.groups.ids='id -G'
alias me.groups.names='id -G -n '
alias me.id='id -u'
alias me.name='id -u -n'

eof
} > ~/.bash_aliases

{ cat << 'eof'

export PATH='/root/.local/bin':$PATH
eof
} >> ~/.bashrc


# system package installs
tmux new -s update -d " \
  apt-get update ;\
  apt-get install -y zstd ;\
  apt-get install -y tree jq ncal less texlive-xetex pandoc ; \
  echo == Done ; \
  sleep 30
"


### Jupyter

In [3]:
%%capture output_install_run_jupyter
%%bash

# jupyter install
tmux new -s jupyter-server -d " \
  pip install ipyaml jupyterlab ; \
  jupyter labextension disable @jupyterlab/apputils-extension:announcements ; \
  jupyter lab \
    --ip=127.0.0.1 \
    --port=8888 \
    --no-browser \
    --allow-root \
    --NotebookApp.token='' ; \
  echo == Done ; \
  sleep 30
"


### Ollama


In [4]:
%%capture output_install_run_ollama
%%bash

# ollama service install and launch
tmux new -s ollama -d "\
  mkdir -p /tmp/ollama-logs/ ; \
  exec > /tmp/ollama-logs/ollama.log 2>&1 ; \
  until which zstd ; do sleep 1 ;done ; \
  curl -fsSL https://ollama.com/install.sh | sh ; \
  OLLAMA_KEEP_ALIVE=20m OLLAMA_FLASH_ATTENTION=1 ollama serve ; \
  echo == Done ; \
  sleep 10
"


### Ollama models


Create a model file to modify the existing model


In [5]:
%%writefile Modelfile.gemma4
FROM gemma4:e4b
PARAMETER num_ctx 32768


Writing Modelfile.gemma4


In [6]:
%%capture output_install_ollama_models
%%bash

# ollama models pull and load
tmux new -s ollama_models -d "\
  mkdir -p /tmp/ollama-logs/ ; \
  exec > /tmp/ollama-logs/ollama.models.log 2>&1 ; \
  until curl -s -I 127.0.0.1:11434 ; do date ; sleep 1 ; done ;\
  ollama create gemma4:e4b-32k -f Modelfile.gemma4 ;\
  ollama ps ; \
  echo llama3.1:8b gemma4:12b  | nice -n 19 ionice -c 3 xargs -n 1 -P 2 ollama pull ; \
  echo ; \
  echo == Done ; \
  sleep 10
"


## Modules, etc.


In [7]:
%alias tree tree

from datetime import datetime, timezone
from time import sleep
from google.colab import output
import requests

print("Waiting for Jupyter to start")
for i in range(300):
  try:
    requests.head( "http://127.0.0.1:8888" )
    print()
    break
  except:
    print("=", end="")
  sleep(1)
print(f"{i} seconds")

print("Jupyter has started")
output.serve_kernel_port_as_window(8888)


Waiting for Jupyter to start
28 seconds
Jupyter has started
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

Wait for Ollama to download a model


In [12]:
%%bash
echo Waiting for a model
date
until curl -s -I 127.0.0.1:11434 ; do sleep 1 ; done
date
until ollama list | grep gemma4:e4b-32k ; do sleep 1 ; done
date


Waiting for a model
Tue Jul 28 12:51:50 AM UTC 2026
HTTP/1.1 200 OK
Content-Type: text/plain; charset=utf-8
Date: Tue, 28 Jul 2026 00:51:50 GMT
Content-Length: 17

Tue Jul 28 12:51:50 AM UTC 2026
gemma4:e4b-32k    92c409fa5c1b    9.6 GB    Less than a second ago    
Tue Jul 28 12:52:55 AM UTC 2026


## Using Open Code and Ollama


When ready, click on the link to Jupyter Lab, open a terminal, and type this to interact with ollama chat:

```
ollama run gemma4:e4b-32k "what is the capital of France?"
```

Or you can run it here.






In [25]:
!ollama run gemma4:e4b-32k "what is the capital of France?"

The capital of France is **Paris**.



In [14]:
!date


Tue Jul 28 12:57:24 AM UTC 2026


## Timer 1


In [15]:
# show a timer for 30 minutes
print("Timer 1")
for i in range(60*30):
  utc_now = datetime.now(timezone.utc)
  print(f"\r{utc_now.timetz().isoformat(timespec='seconds')} {'==' * (utc_now.second % 10)}", end='')
  sleep(1)


Timer 1
01:08:03+00:00 ======

KeyboardInterrupt: 

## Attach Google Drive for data


In [ ]:
from google.colab import drive
drive.mount(
  '/content/drive',
  readonly=True,
)

## Python tool calling


In [17]:
%%bash
pip install ollama --break-system-packages


In [18]:
import ollama
import os


## Autoregressive loop


In [31]:
"""
Token-by-token generation demo using Ollama's raw HTTP API.

Uses requests directly (rather than the ollama client library) so we can see
exactly what's sent/received and rule out any client-side option handling.

Key fix vs. the original script: fully deterministic (greedy) decoding
requires pinning down ALL sampling knobs, not just temperature:
  - temperature: 0      -> disables randomness in scaling logits
  - top_k: 1             -> only ever consider the single highest-prob token
  - top_p: 1.0            -> don't let nucleus sampling filter anything
  - repeat_penalty: 1.0   -> don't penalize repeated tokens (keeps math simple)
  - seed: fixed int      -> in case any residual RNG is used internally
Leaving any of these at model/Modelfile defaults can reintroduce sampling
behavior even with temperature=0.
"""

import requests
import numpy as np
import time
import json

HOST = "http://localhost:11434"
MODEL = "gemma4:e4b-32k"
PROMPT = "What is the capital of France?"
MAX_STEPS = 24
EOS_STRINGS = {"<eos>", "<|endoftext|>", "<|end_of_text|>", "<|im_end|>", ""}

# Fully deterministic decoding options
GEN_OPTIONS = {
    "num_predict": 1,
    "temperature": 0,
    "top_k": 1,
    "top_p": 1.0,
    "repeat_penalty": 1.0,
    "seed": 42,
}


def generate_one_token(prompt: str) -> dict:
    """Call /api/generate for exactly one token, with logprobs."""
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": GEN_OPTIONS,
        "logprobs": True,
        "top_logprobs": 5,
    }
    resp = requests.post(f"{HOST}/api/generate", json=payload, timeout=60)
    resp.raise_for_status()
    return resp.json()


def main():
    current_prompt = PROMPT
    accumulated_text = ""

    print(f"Initial Prompt: \"{PROMPT}\"")
    print(f"Decoding options: {json.dumps(GEN_OPTIONS)}\n")

    for step in range(MAX_STEPS):
        data = generate_one_token(current_prompt)

        chosen_token = data.get("response", "")
        done = data.get("done", False)
        done_reason = data.get("done_reason", "")

        logprobs_block = data.get("logprobs")
        candidates = []
        if logprobs_block and len(logprobs_block) > 0:
            candidates = logprobs_block[0].get("top_logprobs", [])

        is_eos = done or chosen_token in EOS_STRINGS or not chosen_token

        print(f"Step {step + 1}")
        print("-" * 50)
        print(f"Input Context:\n\"{current_prompt}\"\n")
        print(f"Raw done={done} done_reason={done_reason!r} chosen_token={chosen_token!r}\n")

        if candidates:
            print("Next Token Probabilities:")
            print(f"{'Token':<20} | {'Probability':<12}")
            print("-" * 35)
            for item in candidates:
                token_name = item.get("token", "")
                display_name = "[EOS]" if (token_name in EOS_STRINGS or not token_name) else repr(token_name)
                prob_pct = np.exp(item["logprob"]) * 100
                marker = "<-- SELECTED" if token_name == chosen_token else ""
                print(f"{display_name:<20} | {prob_pct:6.2f}%  {marker}")
        else:
            print("(No logprob candidates returned this step.)")

        print(f"\nAccumulated Output: {PROMPT}{accumulated_text}{chosen_token}")
        print("=" * 50 + "\n")

        if is_eos:
            print(f"Generation halted: EOS / done (reason={done_reason!r}).")
            break

        accumulated_text += chosen_token
        current_prompt += chosen_token
        time.sleep(0.5)


main()


Initial Prompt: "What is the capital of France?"
Decoding options: {"num_predict": 1, "temperature": 0, "top_k": 1, "top_p": 1.0, "repeat_penalty": 1.0, "seed": 42}

Step 1
--------------------------------------------------
Input Context:
"What is the capital of France?"

Raw done=True done_reason='length' chosen_token='The'

Next Token Probabilities:
Token                | Probability 
-----------------------------------
'The'                |  77.92%  <-- SELECTED
[EOS]                |  11.43%  
'Paris'              |  10.57%  
'**'                 |   0.07%  
'P'                  |   0.00%  

Accumulated Output: What is the capital of France?The

Generation halted: EOS / done (reason='length').
